In [ ]:
import pandas as pd
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers

# 1. Загрузка данных из Parquet
# Замените 'path_to_file.parquet' на ваш путь, а 'text' — на название колонки с текстом
df = pd.read_parquet('/content/train-00000-of-00001 (1).parquet')
texts = df['text'].astype(str).tolist()

# Создаем генератор, так как tokenizer.train_from_iterator принимает итератор строк
def batch_iterator(batch_size=1000):
    for i in range(0, len(texts), batch_size):
        yield texts[i : i + batch_size]

# 2. Инициализация базового BPE
tokenizer = Tokenizer(models.BPE(unk_token="<unk>", fuse_unk=True))

# 3. Нормализация (Важно для бурятских букв Ү ү, Ө ө, Һ һ)
# StripAccents убран, чтобы гарантированно не повредить Юникод специфичных символов
tokenizer.normalizer = normalizers.Sequence([
    normalizers.NFC()  # Приводим Юникод к единому каноническому виду
])

# 4. Разбивка по словам и пробелам
# Используем связку Whitespace() и Punctuation(), чтобы знаки препинания
# отделялись от бурятских слов и не склеивались с суффиксами
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation()
])

# 5. Настройка тренера (Оптимизировано под агглютинативную структуру бурятского)
trainer = trainers.BpeTrainer(
    vocab_size=16_000,          # Оптимальный размер для выделения чистых корней и суффиксов
    min_frequency=3,            # Порог частоты (можно поднять до 5, если датасет огромный)
    special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
)

# 6. Запускаем обучение прямо из памяти (через итератор)
print("Начало обучения бурятского токенизатора...")
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
print("Обучение завершено!")

# 7. Сохраняем результат
tokenizer.save("my_buryat_tokenizer.json")
print("Токенизатор успешно сохранен в 'my_buryat_tokenizer.json'")

# --- БОНУС: Проверка работы токенизатора ---
# Вы можете сразу протестировать, как он делит сложные бурятские слова
test_phrase = "Буряад Республика уласай нийслэл хото Улаан-Үдэ."
encoded = tokenizer.encode(test_phrase)

print("\nПроверка токенизации:")
print(f"Исходный текст: {test_phrase}")
print(f"Токены: {encoded.tokens}")
print(f"ID токенов: {encoded.ids}")


Начало обучения бурятского токенизатора...
Обучение завершено!
Токенизатор успешно сохранен в 'my_buryat_tokenizer.json'

Проверка токенизации:
Исходный текст: Буряад Республика уласай нийслэл хото Улаан-Үдэ.
Токены: ['ĠÐĳÑĥÑĢÑıÐ°Ð´', 'ĠÐłÐµÑģÐ¿ÑĥÐ±Ð»Ð¸', 'ÐºÐ°', 'ĠÑĥÐ»Ð°ÑģÐ°Ð¹', 'ĠÐ½', 'Ð¸Ð¹', 'Ñģ', 'Ð»ÑįÐ»', 'ĠÑħÐ¾ÑĤÐ¾', 'ĠÐ£Ð»Ð°Ð°Ð½', '-', 'Ò®Ð´Ñį', '.']
ID токенов: [921, 8363, 1123, 4078, 235, 1182, 222, 2600, 2777, 1652, 17, 2848, 18]
